In [3]:
%pip install kagglehub

     ---------------------------------------- 68.3/68.3 KB 1.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\sjune\AppData\Local\Programs\Python\Python39\python.exe -m pip install --upgrade pip' command.


In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("misrakahmed/vegetable-image-dataset")

print("Path to dataset files:", path)

c:\Users\sjune\AppData\Local\Programs\Python\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Resuming download from 17825792 bytes (542205640 bytes left)...
Resuming download from https://www.kaggle.com/api/v1/datasets/download/misrakahmed/vegetable-image-dataset?dataset_version_number=1 (17825792/560031432) bytes left.


100%|██████████| 534M/534M [00:24<00:00, 21.8MB/s] 

Extracting files...


Path to dataset files: C:\Users\sjune\.cache\kagglehub\datasets\misrakahmed\vegetable-image-dataset\versions\1


# 데이터 전처리

In [ ]:
from pathlib import Path

import numpy as np
from PIL import Image
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

dataset_dir = Path(path) / "Vegetable Images"
image_size = (16, 16)
class_names = sorted(folder.name for folder in (dataset_dir / "train").iterdir() if folder.is_dir())
class_to_index = {name: index for index, name in enumerate(class_names)}


def load_split(split):
    features = []
    labels = []
    for class_name in class_names:
        class_dir = dataset_dir / split / class_name
        for image_path in sorted(class_dir.iterdir()):
            if image_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
                continue
            with Image.open(image_path) as image:
                resized = image.convert("RGB").resize(image_size, Image.Resampling.BILINEAR)
                features.append(np.asarray(resized, dtype=np.float32).reshape(-1) / 255.0)
            labels.append(class_to_index[class_name])
    return np.asarray(features, dtype=np.float32), np.asarray(labels, dtype=np.int64)


X_train, y_train = load_split("train")
X_test, y_test = load_split("test")
results = {}


def evaluate_model(name, model):
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    results[name] = accuracy_score(y_test, predictions)
    print(f"{name} accuracy: {results[name]:.4f}")


print(f"Train: {X_train.shape}, Test: {X_test.shape}, Classes: {len(class_names)}")

# Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

logistic_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42),
)
evaluate_model("Logistic Regression", logistic_model)

# k-Nearest Neighbors (k-NN)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5, n_jobs=-1))
evaluate_model("k-NN", knn_model)

# Support Vector Machine (SVM)

In [ ]:
from sklearn.svm import SVC

svm_model = make_pipeline(StandardScaler(), SVC(kernel="rbf"))
evaluate_model("SVM", svm_model)

# Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

decision_tree_model = DecisionTreeClassifier(random_state=42)
evaluate_model("Decision Tree", decision_tree_model)

# Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

random_forest_model = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
evaluate_model("Random Forest", random_forest_model)

# Bagging Classifier

In [ ]:
from sklearn.ensemble import BaggingClassifier

bagging_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(), n_estimators=10, n_jobs=-1, random_state=42
)
evaluate_model("Bagging Classifier", bagging_model)

# Extra Trees

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

extra_trees_model = ExtraTreesClassifier(n_estimators=100, n_jobs=-1, random_state=42)
evaluate_model("Extra Trees", extra_trees_model)

# AdaBoost

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

adaboost_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1), n_estimators=50, random_state=42
)
evaluate_model("AdaBoost", adaboost_model)

# Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gradient_boosting_model = GradientBoostingClassifier(n_estimators=50, random_state=42)
evaluate_model("Gradient Boosting", gradient_boosting_model)

# Perceptron

In [ ]:
from sklearn.linear_model import Perceptron

perceptron_model = make_pipeline(StandardScaler(), Perceptron(max_iter=1000, random_state=42))
evaluate_model("Perceptron", perceptron_model)

# Multi-Layer Perceptron (MLP)

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp_model = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(64,), max_iter=30, early_stopping=True, random_state=42),
)
evaluate_model("MLP", mlp_model)

# Deep Neural Network (DNN)

In [ ]:
# A deeper fully connected network, implemented with scikit-learn's MLPClassifier.
dnn_model = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(64, 32, 16), max_iter=30, early_stopping=True, random_state=42),
)
evaluate_model("DNN", dnn_model)

# Model Accuracy Comparison

In [ ]:
for model_name, accuracy in sorted(results.items(), key=lambda item: item[1], reverse=True):
    print(f"{model_name}: {accuracy:.4f}")